> 对应 Lec 9-11（分类模型部分）。覆盖 Logistic 回归的数值稳定实现、分布式梯度下降/牛顿法/L-BFGS 三种优化器代码、数据缓存模式、X^TWX 广播技巧、分类准确率计算。优化器概念见"6. 光滑优化_概念解释.md"；梯度/Hessian 推导见"5. 分类问题_概念解释.md"。

---

## Logistic 回归数据预处理与缓存模板

**算法思路**：分布式 Logistic 回归需要多轮迭代，每轮都要用到数据 $X$ 和 $y$。若每轮都重传数据，通信量 = 迭代次数 × 数据大小（可能 GB 级），远大于传参数 $\beta$（$p$ 维向量，KB 级）。正确做法是**初始化时一次性把数据推入 Ray 共享内存**，之后只传 $\beta$。

### 添加截距列（np.hstack 全 1 列）

Logistic 回归模型 $P(Y=1|x) = \sigma(x^T\beta)$ 中，$x$ 需要包含常数 1 以引入截距 $\beta_0$。用 `np.hstack` 在特征矩阵左侧拼接全 1 列：

In [ ]:
import numpy as np

# 原始特征矩阵 X_raw: (n, p_raw)，不含截距
X_raw = np.random.randn(1000, 50)

# 添加截距列（全 1），结果 X: (n, p_raw+1)
X = np.hstack([np.ones((X_raw.shape[0], 1)), X_raw])
# np.ones((n, 1)) 创建 n×1 的全 1 列，hstack 水平拼接
# 此后 beta[0] 对应截距，beta[1:] 对应各特征系数

### 数据一次性缓存到 Ray 内存（X_refs / y_refs 模式）

In [ ]:
import ray, pandas as pd

ray.init()

X_refs = []   # 存放所有数据块的 ObjectRef
y_refs = []

# 流式读取 + 一次性缓存（只做一次数据传输）
for chunk in pd.read_table(
    "data/sim_data.txt",
    skiprows=2,       # 跳过前 2 行注释
    delimiter=";",    # 分隔符为分号
    header=None,
    chunksize=100000
):
    arr = chunk.values              # 转 ndarray
    y_chunk  = arr[:, 0]           # 第 0 列是标签 Y
    X_raw    = arr[:, 2:]          # 第 2 列起是特征 X（跳过 Y2）
    X_chunk  = np.hstack([np.ones((len(y_chunk), 1)), X_raw])   # 添加截距

    X_refs.append(ray.put(X_chunk))   # 永久缓存到集群内存
    y_refs.append(ray.put(y_chunk))

p = X_chunk.shape[1]   # 参数维度（含截距）
print(f"共 {len(X_refs)} 个数据块，参数维度 p={p}")

### 为什么不在每轮迭代时重传数据（通信 vs 计算开销对比）

```
每轮迭代的数据量（假设 n=10^6，p=50，float64）：
  数据：10^6 × 51 × 8 = ~400MB / 块，10 块共 ~4GB
  参数 β：51 × 8 = ~400字节

正确做法（只传 β）：每轮通信 400字节 × 10节点 = 4KB
错误做法（重传数据）：每轮通信 4GB，100轮迭代 = 400GB 网络流量
```

---

## 数值稳定的 Sigmoid（stable_sigmoid）

**算法思路**：朴素实现 $\sigma(z) = 1/(1+e^{-z})$，当 $z > 710$ 时 $e^{-z}$ 下溢为 0（安全），但若用 $e^z/(1+e^z)$ 形式当 $z > 710$ 时 $e^{710}$ 溢出为 `inf`。分段技巧：无论 $z$ 正负，总是计算 $e^{-|z|}$，该值永远 $\leq 1$，不会溢出。

### 朴素实现的溢出场景（e^710 → inf）

In [ ]:
# 朴素实现
def naive_sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))   # z 极大时 exp(-z)=0，安全
                                        # z 极小时 exp(-z)→inf，结果=0，也安全
# 实际上朴素 sigmoid 对单纯的 z 是安全的，但 z < 0 时
# 改用 exp(z)/(1+exp(z)) 形式就会溢出

z = np.array([-800.0, 800.0])
print(np.exp(-np.abs(z)))   # [0, 0]，永不溢出
print(np.exp(z))            # [0, inf]，正数时溢出

### 分段处理：np.where + np.exp(-np.abs(x)) 实现

In [ ]:
def stable_sigmoid(z):
    """
    数值稳定的 Sigmoid，避免 exp 溢出。
    原理：用 e^{-|z|} 作为基础（永不溢出），分段处理 z>=0 和 z<0 两种形式。
    """
    e = np.exp(-np.abs(z))         # e^{-|z|}，永远 <= 1，不溢出
    numer = np.where(z >= 0, 1.0, e)   # z>=0 时分子=1，z<0 时分子=e^z
    denom = 1.0 + e                    # 分母=1+e^{-|z|}，两种情况统一
    return numer / denom
    # z>=0 时：1 / (1+e^{-z}) = σ(z)   ✓
    # z<0 时：e^z / (1+e^z) = σ(z)     ✓，且指数在分子，z<0 则 e^z<1，安全

# 验证极端值
print(stable_sigmoid(np.array([-1000.0, 0.0, 1000.0])))
# 输出：[~0, 0.5, ~1]，不出现 nan 或 inf

### 生产用法：scipy.special.expit(z)

In [ ]:
from scipy.special import expit

# scipy 的 expit 就是 sigmoid，底层 C 实现，数值稳定且更快
p = expit(X @ beta)   # 在所有分布式代码中推荐直接用这个

# expit 支持标量、向量、矩阵
print(expit(np.array([-1000.0, 0.0, 1000.0])))  # [~0, 0.5, ~1]

---

## 数值稳定的 Softplus（stable_softplus）

**算法思路**：Softplus $f(z) = \log(1+e^z)$，直接计算当 $z$ 很大时 $e^z$ 溢出。等价变形 $\log(1+e^z) = \max(z,0) + \log(1+e^{-|z|})$，其中 $e^{-|z|} \leq 1$ 永不溢出。Logistic 回归损失用 Softplus 形式 $\ell = -yz + \text{softplus}(z)$ 避免对概率取对数（防止 $\log 0$）。

### 等价变形：log(1+e^z) = max(z,0) + log(1+e^{-|z|})

In [ ]:
# 推导：
# z >= 0 时：log(1+e^z) = log(e^z(e^{-z}+1)) = z + log(1+e^{-z})
#            = max(z,0) + log(1+e^{-|z|})
# z < 0 时：log(1+e^z) = 0 + log(1+e^z)
#            = max(z,0) + log(1+e^{-|z|})（因为 z<0 时 |z|=-z，e^{-|z|}=e^z）

### 代码实现（np.maximum + np.log）

In [ ]:
def stable_softplus(z):
    """
    数值稳定的 log(1+e^z)。
    关键：e^{-|z|} <= 1，log(1+e^{-|z|}) 的参数在 (1,2] 之间，安全。
    """
    e = np.exp(-np.abs(z))                     # 永不溢出
    return np.maximum(z, 0.0) + np.log(1.0 + e)

# 验证
print(stable_softplus(np.array([-1000.0, 0.0, 1000.0])))
# 输出：[~0, 0.693, 1000]，不出现 inf

### 生产用法：scipy.special.softplus(z)

In [ ]:
from scipy.special import softplus

# 直接调用，底层数值稳定实现
loss_terms = -y * z + softplus(z)   # Logistic 回归损失的向量化写法
total_loss = loss_terms.sum()

### 损失函数用 Softplus 形式而非 log(p) 的原因

In [ ]:
# ❌ 原始形式：数值不稳定
loss_naive = -y * np.log(p) - (1 - y) * np.log(1 - p)
# 当 p → 0 或 p → 1 时，log(p) 或 log(1-p) → -inf → nan

# ✅ Softplus 形式：等价且数值稳定
z = X @ beta
loss_stable = -y * z + softplus(z)
# 完全绕开对 p 取对数，使用线性变量 z = x^Tβ 进行计算

---

## X^TWX 广播计算（不构造 n×n 的 W）

**算法思路**：牛顿法 Hessian $H = X^TWX$，其中 $W = \text{diag}(w)$，$w_i = p_i(1-p_i)$。$W$ 是 $n\times n$ 矩阵，当 $n=10^6$ 时需要 8TB 内存。用广播乘法代替：$WX$ 等价于对 $X$ 每行乘以对应的权重 $w_i$，即 `X * w.reshape(-1, 1)`。

### w = p*(1-p) 对角权重向量

In [ ]:
from scipy.special import expit

beta = np.zeros(p)
z = X @ beta
p_prob = expit(z)               # 预测概率，shape (n,)

# Hessian 的对角权重 w_i = p_i(1-p_i)
w = p_prob * (1.0 - p_prob)    # shape (n,)，所有元素 > 0（p_i ∈ (0,1)）
# 注意：w_i 最大在 p_i=0.5 时取 0.25，在 p_i→0 或 p_i→1 时趋近 0

### 正确写法：X.T @ (X * w.reshape(-1, 1))

In [ ]:
# w.reshape(-1, 1) 将 (n,) 变为 (n, 1)，与 (n, p) 广播后每行乘 w_i
# X * w.reshape(-1, 1) 等价于 W @ X（W = diag(w)），结果 shape (n, p)
# X.T @ (W @ X) = X^T W X

hess = X.T @ (X * w.reshape(-1, 1))   # shape (p, p)，对称正定

### 错误写法：np.diag(w) 构造完整矩阵（内存爆炸）

In [ ]:
# ❌ 绝对不要这么写：
W_matrix = np.diag(w)             # n=100万时需要 8TB 内存，直接 MemoryError
hess_wrong = X.T @ W_matrix @ X  # 即使没崩溃也极慢

# 牢记：对角矩阵乘以向量/矩阵，永远用广播代替 np.diag()

---

## 分布式梯度下降法（Logistic 回归）

**算法思路**：梯度 $\nabla\ell(\beta) = X^T(p-y)$ 具有可加性，各节点持有数据块 $X_i, y_i$，计算局部梯度 $X_i^T(p_i - y_i)$，主节点求平均得全局梯度，更新 $\beta$。迭代时只传 $\beta$（小向量），数据留在各节点共享内存。

### 局部梯度 chunk 函数（grad + loss + ni 三元组返回）

In [ ]:
import ray
from scipy.special import expit, softplus

@ray.remote
def compute_gradient_chunk(X_ref, y_ref, beta):
    """
    在一块数据上计算局部梯度、局部损失和样本量。
    返回三元组，主节点用于 Reduce 汇总。
    """
    X_chunk = X_ref            # ObjectRef 在 Worker 进程中自动解引用
    y_chunk = y_ref
    z     = X_chunk @ beta     # 线性预测值，shape (n,)
    p     = expit(z)           # 数值稳定的 sigmoid
    grad  = X_chunk.T @ (p - y_chunk)              # 局部梯度，shape (p,)
    loss  = np.sum(-y_chunk * z + softplus(z))     # 局部损失（softplus 形式）
    ni    = X_chunk.shape[0]   # 局部样本量（用于计算平均梯度）
    return grad, loss, ni

### 主节点 Reduce：全局梯度 = Σgradᵢ / N

In [ ]:
def distributed_gd_step(X_refs, y_refs, beta):
    """一次梯度下降迭代（Map-Reduce）"""
    tasks = [compute_gradient_chunk.remote(Xr, yr, beta)
             for Xr, yr in zip(X_refs, y_refs)]
    results = ray.get(tasks)   # Barrier

    N           = sum(r[2] for r in results)
    global_grad = sum(r[0] for r in results) / N   # 平均梯度（normalize 到每样本）
    global_loss = sum(r[1] for r in results) / N
    return global_grad, global_loss

### 参数更新：β ← β - lr * global_grad

In [ ]:
def distributed_gradient_descent(X_refs, y_refs, p, lr=0.1, n_iters=100):
    beta = np.zeros(p)    # 初始化为 0

    for t in range(n_iters):
        global_grad, global_loss = distributed_gd_step(X_refs, y_refs, beta)
        beta = beta - lr * global_grad   # 梯度下降更新

        if (t + 1) % 10 == 0:
            print(f"Iter {t+1:3d}: loss={global_loss:.6f}, "
                  f"|grad|={np.linalg.norm(global_grad):.2e}")

        if np.linalg.norm(global_grad) < 1e-6:
            print(f"收敛于第 {t+1} 次迭代")
            break

    return beta

### 每轮只传小的 β 向量（而非重传数据）

In [ ]:
# 迭代循环中，只传 beta 给 .remote()（p 维 float64 向量，约 400 字节）
# X_refs 和 y_refs 是 ObjectRef，传的是引用 ID（几十字节），不传数据本身
# 数据只在初始化时传了一次，存放在 Ray 集群的共享内存中
tasks = [compute_gradient_chunk.remote(Xr, yr, beta) for Xr, yr in zip(X_refs, y_refs)]
#                                              ^^^^
#                                    每轮只传这个小向量

---

## 分布式牛顿法（Logistic 回归）

**算法思路**：牛顿法用二阶信息（Hessian）自动确定步长和方向，收敛极快（二阶收敛，通常 10-30 次迭代）。代价是每轮需计算 $p\times p$ 的 Hessian（通信量增加），并在主节点解 $p\times p$ 的线性方程组。适合 $p$ 中等（百到千量级）、追求快速收敛的场景。

### 局部梯度 + Hessian chunk 函数（grad + hess + loss + ni）

In [ ]:
import scipy.linalg

@ray.remote
def compute_grad_hess_chunk(X_ref, y_ref, beta):
    """计算局部梯度和 Hessian"""
    from scipy.special import expit, softplus

    X_chunk = X_ref
    y_chunk = y_ref
    z     = X_chunk @ beta
    p     = expit(z)

    grad  = X_chunk.T @ (p - y_chunk)    # 局部梯度

    # 高效计算 X_i^T W_i X_i（广播，不构造对角矩阵）
    w     = p * (1.0 - p)               # 对角权重，shape (n,)
    hess  = X_chunk.T @ (X_chunk * w.reshape(-1, 1))   # 局部 Hessian (p,p)

    loss  = np.sum(-y_chunk * z + softplus(z))
    ni    = X_chunk.shape[0]
    return grad, hess, loss, ni

### Hessian 加对角正则化（+1e-6 * I）防奇异

In [ ]:
def distributed_newton_step(X_refs, y_refs, beta, p):
    tasks = [compute_grad_hess_chunk.remote(Xr, yr, beta)
             for Xr, yr in zip(X_refs, y_refs)]
    results = ray.get(tasks)

    N           = sum(r[3] for r in results)
    global_grad = sum(r[0] for r in results) / N
    global_hess = sum(r[1] for r in results) / N

    # 对角加小正则防奇异（Hessian 理论上半正定，加正则使严格正定）
    # 确保 Cholesky 不出错（Cholesky 要求严格正定）
    global_hess += np.eye(p) * 1e-6

    global_loss = sum(r[2] for r in results) / N
    return global_grad, global_hess, global_loss

### 用 Cholesky 解牛顿方向（H 是对称正定矩阵）

In [ ]:
def distributed_newton(X_refs, y_refs, p, tol=1e-6, max_iters=50):
    beta = np.zeros(p)

    for t in range(max_iters):
        grad, hess, loss = distributed_newton_step(X_refs, y_refs, beta, p)

        # Hessian 是对称正定矩阵，用 Cholesky 比 LU 快约 2×
        cho_fac = scipy.linalg.cho_factor(hess, lower=True)
        # 解方程 H * delta = grad，得到牛顿方向 delta = H^{-1} g
        delta = scipy.linalg.cho_solve(cho_fac, grad)

        beta = beta - delta    # 牛顿步，天然步长 α=1（不需要调学习率）

        if np.linalg.norm(grad) < tol:
            print(f"收敛于第 {t+1} 次迭代")
            break

    return beta

### 牛顿步更新：β ← β - H^{-1}g（天然步长 α=1）

牛顿法的更新量 $\delta = H^{-1}g$ 已经包含了曲率信息，步长 $\alpha=1$ 是理论上的最优步长（在强凸函数的邻域内）。与梯度下降不同，**不需要手动调学习率**。

In [ ]:
beta = beta - delta        # ✅ 天然步长 1
beta = beta - 0.1 * delta  # ❌ 通常不这么做，违背牛顿法设计

---

## 分布式 L-BFGS（Logistic 回归）

**算法思路**：L-BFGS 是拟牛顿法，不显式计算存储 Hessian（$O(p^2)$ 内存），而是只保存最近 $m$（默认 5-10）步的梯度变化历史来近似 $H^{-1}$，内存 $O(mp)$。通过 `scipy.optimize.minimize` 封装接口，L-BFGS 自动管理历史和线搜索，代码简洁。分布式时只需要将"计算损失和梯度"的函数接口改为 Ray Map-Reduce 即可。

### scipy.optimize.minimize 接口封装（method='L-BFGS-B'）

In [ ]:
from scipy.optimize import minimize

# L-BFGS 需要一个函数：给定 beta，返回 (loss, grad)
# 只需要把这个函数改为分布式版本，L-BFGS 的其他逻辑由 scipy 处理

def global_objective(beta, X_refs, y_refs):
    """
    L-BFGS 优化器的目标函数接口，scipy 每步都会调用它。
    输入：参数向量 beta
    输出：(损失值, 梯度向量)
    """
    tasks = [compute_gradient_chunk.remote(Xr, yr, beta)
             for Xr, yr in zip(X_refs, y_refs)]
    results = ray.get(tasks)

    N           = sum(r[2] for r in results)
    total_loss  = sum(r[1] for r in results) / N
    total_grad  = sum(r[0] for r in results) / N

    # scipy 要求梯度必须是 float64 的 C-contiguous 数组
    return float(total_loss), total_grad.astype(np.float64)

### 分布式梯度函数作为 jac 参数传入

In [ ]:
from scipy.optimize import minimize
import functools

beta_init = np.zeros(p)

# functools.partial 绑定固定参数（X_refs, y_refs），让函数签名只剩 beta
obj_func = functools.partial(global_objective, X_refs=X_refs, y_refs=y_refs)

result = minimize(
    fun=obj_func,       # 同时返回 loss 和 grad 的函数
    x0=beta_init,
    method='L-BFGS-B',
    jac=True,           # 告诉 scipy：fun 的返回值是 (loss, grad) 的元组
    options={
        'maxiter': 200,         # 最大迭代次数
        'ftol': 1e-10,          # 函数值变化容差
        'gtol': 1e-6,           # 梯度范数容差
        'maxcor': 10,           # L-BFGS 保留历史步数 m（内存 O(mp)）
    }
)

beta_lbfgs = result.x
print(f"收敛: {result.success}，迭代次数: {result.nit}，最终损失: {result.fun:.6f}")

也可以用 `fmin_l_bfgs_b`（旧接口，参数格式略有不同）：

In [ ]:
from scipy.optimize import fmin_l_bfgs_b

def obj_for_fmin(beta):
    return global_objective(beta, X_refs, y_refs)

beta_lbfgs, final_loss, info = fmin_l_bfgs_b(
    func=obj_for_fmin,
    x0=beta_init,
    maxiter=200
)
# info['warnflag'] == 0 表示成功收敛，1 表示超过最大迭代，2 表示其他问题
print(f"warnflag={info['warnflag']}，函数调用次数={info['funcalls']}")

### L-BFGS 与牛顿法的对比（内存 O(mp) vs O(p²)）

| 特性 | 牛顿法 | L-BFGS |
|------|--------|--------|
| Hessian 存储 | $O(p^2)$（完整矩阵） | $O(mp)$（$m$ 步历史，$m\approx 10$） |
| 每步通信量 | $p + p^2$（梯度+Hessian） | $p + 1$（梯度+损失） |
| 收敛速度 | 二阶（极快） | 超线性（快，介于GD和牛顿之间） |
| $p=10000$ 时内存 | $10000^2 \times 8 = 800$MB | $10 \times 10000 \times 8 = 800$KB |
| 适用场景 | $p < 1000$ | $p$ 较大，分布式首选 |

---

## 分类准确率的分布式计算

**算法思路**：预测标签 $\hat y_i = \mathbf{1}[\sigma(x_i^T\hat\beta) \geq 0.5]$，准确率 = 预测正确的样本数 / 总样本数。各节点统计本地正确数和样本量，主节点汇总后计算比率。

### 局部预测（sigmoid(Xβ) > 0.5）+ 局部正确数

In [ ]:
@ray.remote
def compute_accuracy_chunk(X_ref, y_ref, beta):
    """在一块数据上计算预测准确率的局部统计量"""
    from scipy.special import expit
    X_chunk = X_ref
    y_chunk = y_ref

    prob  = expit(X_chunk @ beta)          # 预测概率，shape (n,)
    y_hat = (prob >= 0.5).astype(int)      # 预测标签（阈值 0.5）
    correct = np.sum(y_hat == y_chunk)     # 预测正确的样本数
    return int(correct), len(y_chunk)      # 返回 (正确数, 样本量)

### 全局准确率 = 总正确数 / 总样本数（Reduce）

In [ ]:
def compute_accuracy(X_refs, y_refs, beta):
    """分布式计算准确率"""
    tasks = [compute_accuracy_chunk.remote(Xr, yr, beta)
             for Xr, yr in zip(X_refs, y_refs)]
    results = ray.get(tasks)

    total_correct = sum(r[0] for r in results)
    total_n       = sum(r[1] for r in results)
    return total_correct / total_n   # 不要用 np.mean(各块准确率)！

accuracy = compute_accuracy(X_refs, y_refs, beta_lbfgs)
print(f"预测准确率：{accuracy:.4f}")

# ⚠️ 易错：不能直接平均各块的准确率（各块样本量可能不同）
# 正确做法：汇总总正确数和总样本数，再做比值

完整的分布式 Logistic 回归训练流程（hw4 风格题目）：

In [ ]:
import ray, numpy as np, pandas as pd, scipy.linalg
from scipy.special import expit, softplus
from scipy.optimize import minimize
import functools

ray.init()

# ===== 1. 数据加载与缓存 =====
X_refs, y_refs = [], []
for chunk in pd.read_table("data.txt", skiprows=2, delimiter=";",
                            header=None, chunksize=100000):
    arr    = chunk.values
    y      = arr[:, 0]
    X_raw  = arr[:, 2:]
    X      = np.hstack([np.ones((len(y), 1)), X_raw])
    X_refs.append(ray.put(X))
    y_refs.append(ray.put(y))

p = X.shape[1]

# ===== 2. 定义分布式梯度函数 =====
@ray.remote
def grad_chunk(X_ref, y_ref, beta):
    z    = X_ref @ beta
    p_   = expit(z)
    grad = X_ref.T @ (p_ - y_ref)
    loss = np.sum(-y_ref * z + softplus(z))
    return grad, loss, X_ref.shape[0]

def objective(beta):
    results   = ray.get([grad_chunk.remote(Xr, yr, beta) for Xr, yr in zip(X_refs, y_refs)])
    N         = sum(r[2] for r in results)
    loss      = sum(r[1] for r in results) / N
    grad      = sum(r[0] for r in results) / N
    return float(loss), grad.astype(np.float64)

# ===== 3. L-BFGS 优化 =====
result = minimize(objective, np.zeros(p), method='L-BFGS-B', jac=True,
                  options={'maxiter': 200, 'gtol': 1e-6})
beta_hat = result.x

# ===== 4. 计算准确率 =====
@ray.remote
def acc_chunk(X_ref, y_ref, beta):
    correct = np.sum((expit(X_ref @ beta) >= 0.5) == y_ref.astype(bool))
    return int(correct), X_ref.shape[0]

results  = ray.get([acc_chunk.remote(Xr, yr, beta_hat) for Xr, yr in zip(X_refs, y_refs)])
accuracy = sum(r[0] for r in results) / sum(r[1] for r in results)
print(f"准确率: {accuracy:.4f}")

ray.shutdown()